# NFL week projections: drive-level Monte Carlo

Pulls nflverse drive results (this season, plus last season at a lower weight), builds offense and defense rates for touchdown / field goal / empty drive / defensive score, draws possessions from a Poisson distribution, and simulates every game of the chosen week 10,000 times.

**Run it:** Runtime → Run all. Change `WEEK` below for another slate. Works from a phone browser.

In [ ]:
SEASON = 2026
WEEK = 3
SIMS = 10_000

In [ ]:
import argparse
import time

import numpy as np
import pandas as pd

PBP_URL = "https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{season}.parquet"
GAMES_URL = "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv"

OUTCOMES = ["TD", "FG", "NONE", "DEF_TD", "SAFETY"]
POINTS_FOR = np.array([7, 3, 0, 0, 0])      # points to the offense
POINTS_AGAINST = np.array([0, 0, 0, 7, 2])  # points to the defense
RESULT_MAP = {"Touchdown": "TD", "Field goal": "FG", "Opp touchdown": "DEF_TD", "Safety": "SAFETY"}
KEY_NUMBERS = [3.5, 7.5]


def load_drives(season, prior_weight):
    """One row per drive: offense, defense, outcome, weight (this season 1.0, last season prior_weight)."""
    frames = []
    for year, weight in ((season - 1, prior_weight), (season, 1.0)):
        pbp = pd.read_parquet(PBP_URL.format(season=year),
                              columns=["game_id", "season_type", "posteam", "defteam",
                                       "fixed_drive", "fixed_drive_result"])
        pbp = pbp[(pbp.season_type == "REG") & pbp.posteam.notna() & pbp.fixed_drive_result.notna()]
        d = pbp.groupby(["game_id", "fixed_drive"]).last().reset_index()
        d["outcome"] = d.fixed_drive_result.map(RESULT_MAP).fillna("NONE")
        d["weight"] = weight
        frames.append(d[["game_id", "posteam", "defteam", "outcome", "weight"]])
    return pd.concat(frames, ignore_index=True)


def team_rates(drives, prior_drives):
    """Offense and defense outcome rates, shrunk toward league average by prior_drives."""
    league = drives.groupby("outcome").weight.sum().reindex(OUTCOMES, fill_value=0)
    league = league / league.sum()

    def rates(side):
        counts = drives.pivot_table(index=side, columns="outcome", values="weight",
                                    aggfunc="sum", fill_value=0).reindex(columns=OUTCOMES, fill_value=0)
        shrunk = counts.add(league * prior_drives, axis=1)
        return shrunk.div(shrunk.sum(axis=1), axis=0)

    games = drives.groupby(["game_id", "posteam"]).agg(n=("outcome", "size"), w=("weight", "first"))
    pace = (games.n * games.w).groupby("posteam").sum() / games.w.groupby("posteam").sum()
    return rates("posteam"), rates("defteam"), league, pace


def matchup_probs(off, dfn, league, boost):
    """Log5-style blend of an offense against a defense; boost scales scoring for home field."""
    p = off.values * dfn.values / league.values
    p[:2] *= boost
    return p / p.sum()


def simulate(home, away, off, dfn, league, pace, sims, hfa, rng):
    lam = (pace[home] + pace[away]) / 2
    p_home = matchup_probs(off.loc[home], dfn.loc[away], league, 1 + hfa)
    p_away = matchup_probs(off.loc[away], dfn.loc[home], league, 1 - hfa)
    home_drives = rng.multinomial(rng.poisson(lam, sims), p_home)
    away_drives = rng.multinomial(rng.poisson(lam, sims), p_away)
    home_pts = home_drives @ POINTS_FOR + away_drives @ POINTS_AGAINST
    away_pts = away_drives @ POINTS_FOR + home_drives @ POINTS_AGAINST
    return home_pts, away_pts


def run(season, week, sims=10_000, prior_weight=0.35, prior_drives=40, hfa=0.05, seed=7):
    """Projections for every regular-season game in the given week, as a DataFrame."""
    drives = load_drives(season, prior_weight)
    off, dfn, league, pace = team_rates(drives, prior_drives)
    games = pd.read_csv(GAMES_URL)
    slate = games[(games.season == season) & (games.week == week) & (games.game_type == "REG")]
    rng = np.random.default_rng(seed)

    rows = []
    for g in slate.itertuples():
        hp, ap = simulate(g.home_team, g.away_team, off, dfn, league, pace, sims, hfa, rng)
        margin, total = hp - ap, hp + ap
        row = {
            "matchup": f"{g.away_team} @ {g.home_team}",
            "kickoff": g.gameday,
            "fair_spread_home": round(-margin.mean(), 1),   # e.g. -3.2 = home favored by 3.2
            "market_spread_home": -g.spread_line if pd.notna(g.spread_line) else None,
            "fair_total": round(total.mean(), 1),
            "market_total": g.total_line,
            "home_win_pct": round(100 * ((margin > 0).mean() + 0.5 * (margin == 0).mean()), 1),
        }
        for k in KEY_NUMBERS:
            row[f"home_cover_-{k}"] = round(100 * (margin > k).mean(), 1)
            row[f"away_cover_-{k}"] = round(100 * (margin < -k).mean(), 1)
        if pd.notna(g.total_line):
            row["over_market_pct"] = round(100 * (total > g.total_line).mean(), 1)
        rows.append(row)
    return pd.DataFrame(rows), len(drives)


In [ ]:
import time
t0 = time.time()
df, n_drives = run(SEASON, WEEK, SIMS)
print(f'{len(df)} games x {SIMS:,} sims from {n_drives:,} drives in {time.time() - t0:.1f}s')
df

In [ ]:
# Download the table as CSV
df.to_csv(f'nfl_week{WEEK}_projections.csv', index=False)
try:
    from google.colab import files
    files.download(f'nfl_week{WEEK}_projections.csv')
except ImportError:
    pass

In [ ]:
# Optional: write the table to a new Google Sheet (asks you to sign in to Google)
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create(f'NFL week {WEEK} projections')
sh.sheet1.update([df.columns.tolist()] + df.astype(str).values.tolist())
print(sh.url)